# Lesson 5.1: Basic Sentiment Analysis with VADER

**🎬 Video:** [Lesson 5.1: Basic Sentiment Analysis with VADER](#)

## Overview

Sentiment analysis is a field in machine learning that tries to evaluate the emotional tone of piece of written text. The general goal is to establish whether the text is positive or negative in tone, while some specific models try to capture a range of emotions. In doing so, it is trying to solve one very fundamental problem in data science: the overwhelming majority of data on that contains people's feelings is not packaged in neat surveys, but is offered as raw text. 

A positive comment about a product in a review, a negative social media post about a current event, or an ambivalent description about a medical condition in a case file are all raw text data that contain emotions. While these individual records are usually easily decipherable by humans, they are incredibly hard to quantify. Some products have thousands of reviews and there are billions of social media users posting every day. Sentiment analysis tries to capture *how* people feel towards the things they are writing about. The field has a number of practical applications. Someone in marketting might use it to aggregate reviews of a product, a medical researcher might use it to comb through thousands of descriptions of a condition, and a political strategist might use sentiment analysis on social media data to guage interest in a candidate.

As with geoparsing, getting computers to think like humans is an incredibly difficult task and there are several fundamental limitations to this type of work. Human emotions do not exists as specific moments in time or in a linear manner, but are part of a dynamic system that is constantly being updated through new experiences. As such, it is hard to distinguish where one emotion begins and another ends. A party may be boring, but suddenly becoming interesting or even fun. What's more, the party may be more fun now than it was before, but may still be less fun than another experience, like a previous party or an entirely different event like walking the dog. 

For humans, language always seems to fall short when we try capture these highly fluid and ill-defined states. Loaded words like "love" and "hate" can feel inadequate to the strength of the emotion we are feeling. Conversely, some expressions can overestimate the emotional experience of an event: "I was having the best ice-cream in the world when it fell on the floor. It was traumatic." It is unlikely that the ice-cream was truly the "best" in the world nor is it likely that dropping it would have caused the clinical definition of trauma. Finally, like describing locations in language, emotions in language can be described using colloquial or idiomatic expressions. Depending on what text the sentiment model was trained, expressions like "cringe", "sus", or "extra" might be quite meaningless. Despite these limitations, these sentiment models excel at churning through text at a high rate and establishing the basic contours of the emotional valence.

This lesson will cover two sentiment analysis methods:
- Using the **NLTK** library's VADER sentiment analysis tool.
- Using **Hugging Face's RoBERTa** model for sentiment analysis (Lesson 5_2)

We will compare how these two tools perform on sentences containing toponyms extracted from the `jmu_reddit_geoparsed_cleaned.pickle` file, and we will store the results in a **pandas** DataFrame for further analysis. The key goal is to understand how different tools analyze sentiment, identify their limitations, and explore why their outputs might differ. 

Unlike the `geoparser` you will run the RoBERTa model to get your own results for the final project.

---

## 1 Sentiment Analysis with NLTK (VADER)

### 1.1 Overview
VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analysis tool specifically attuned to sentiments expressed in social media. It was largely trained on Twitter data and evaluates sentiment at the word level. This makes it fast, but it has notable limitations.

The sentiment analyzer is easy to use. Once the libraries are loaded calling the `sia.polarity_scores` function on any text performs the analysis. For example:

```python
sia.polarity_scores('I love going to Buc-ees, but I do not love the gas prices there.')
```

Output:
```
{'neg': 0.0, 'neu': 0.543, 'pos': 0.457, 'compound': 0.8555}
```

The four scores are:

| Score | Range | Meaning |
|---|---|---|
| `neg` | 0.0 – 1.0 | Proportion of text that is negative |
| `neu` | 0.0 – 1.0 | Proportion of text that is neutral |
| `pos` | 0.0 – 1.0 | Proportion of text that is positive |
| `compound` | -1.0 – +1.0 | Overall sentiment: < -0.05 = negative, > 0.05 = positive, in between = neutral |

### 1.2 Evaluating Positive Scores

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize the VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()
sia.polarity_scores("I love going to Buc-ees, but I do not love the gas prices there.")

> 💡 **Reflection:** The positive score is `0.45` and the neutral score is `0.54`. 
> - What parts of the sentence might the algorithm be seeing as positive, and what parts as neutral? 
> The overall compound score is `0.86`. 
> - Do you agree with this assessment? Is it too positive, or too negative?

### 1.3 Evaluating Negative Scores

The sentence below shares the same negation principle as the above sentence where `not` is being used to create the opposite of a particular feeling.

In [ ]:
sia.polarity_scores("UVA is not the best university!")

> 💡 **Reflection:** What do you make of this assessment? Why is this score so much more negative than the above sentence which was overwhelmingly positive?

### ✍️ 1.4 Critical Activity - Making Emotions

For the next activity, you are going to try to push the limits of the tokenizer. For each challenge, think of a sentence that will get the scores you want, even if those scores don't make sense.

#### Challenge 1: Most Goodest Vibes

Create a sentence with a compound polarity score of 1.0.

In [ ]:
sia.polarity_scores("")

#### Challenge 2: Most Baddest Vibes

 Create a sentence with a compound polarity score of -1.0, but keep it PG-13 so you can share with the class!

In [ ]:
sia.polarity_scores("")

#### Challenge 3: Most Strangest Vibes

Create a sentence with either a positive or negative compound score, but that means the exact opposite of what it says.

In [ ]:
sia.polarity_scores("")

> 💡 **Reflection:** What were some ways you manipulated the tokenizer to get the result you wanted? How does this show the limits of what the tokenizer is capable of?

### 1.5 Run and Evaluate VADER

While VADER is relatively quick and easy to implement, some of the quirks of how it evaluates sentiment become visible when applied to a large dataset. Having been trained on Twitter it excels at passages that lack nuance, but has a much harder time with complicated human emotions.

The code below runs the analysis on all lines and produces a table with outputs.

In [ ]:
import os
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize the VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()
_primary = "../data/JMU/JMU_geoparsed_cleaned.pkl"
_backup = "../data/JMU/JMU_geoparsed_long_cleaned_backup.csv"

if os.path.exists(_primary):
    df_reddit_geoparsed_long = pd.read_pickle(_primary)
else:
    df_reddit_geoparsed_long = pd.read_csv(_backup)
    print("⚠️  JMU_geoparsed_cleaned.pkl not found — using backup (complete Lesson 4.4 to use your team's cleaned data)")

# Perform sentiment analysis on each sentence and store the compound score in the DataFrame
df_reddit_geoparsed_long["vader_sentiment"] = df_reddit_geoparsed_long["sentences"].apply(lambda x: sia.polarity_scores(x)["compound"])
(df_reddit_geoparsed_long[["sentences", "vader_sentiment"]]
    .sample(5, random_state=7)
    .style
    .set_properties(subset=["sentences"], **{"white-space": "normal", "width": "600px"})
    .set_properties(subset=["vader_sentiment"], **{"width": "120px", "text-align": "center"})
    .format({"vader_sentiment": "{:.3f}"})
)

The compound score ranges from -1 to 1. When a passage is very negative it gets a -1 and when it is possitive it gets a 1. Read through the passages above and try to figure out why these passages received the sentiments they did.

> 💡 **Critical Reflection:** How effective is the VADER tokenizer in dealing with sentiments?
> - Look at the sample output above. Are there any sentences where the score surprises you?
> - VADER was trained on Twitter data. Does that seem to match the tone of Reddit posts?
> - What kinds of sentences do you think would consistently fool a rule-based tool like VADER?


---

## Lesson Summary

Here is what you covered in this lesson:

### Part 1: Rule-Based Sentiment Analysis with VADER
- **VADER** — a rule-based tool trained on social media text; scores sentiment at the word level
- **`sia.polarity_scores(text)`** — returns `neg`, `neu`, `pos`, and `compound` scores for any string
- **`compound`** — the overall sentiment score, ranging from -1.0 (very negative) to +1.0 (very positive)
- **Limitations** — VADER struggles with negation, sarcasm, and complex emotional nuance; it treats every word independently

---

In the next lesson you will run a transformer-based model (RoBERTa) on the same data and compare its results to VADER.

➡️ **Next:** [Lesson 5.2 — Sentiment Analysis with RoBERTa](lesson_5_2_roberta_sentiment.ipynb)
